```text
QDM Synthetic Data + 3D Inversion + Magnetic source location, direction, intensity Detection

 This script performs the full workflow:
 1. Create synthetic magnetic observations (dipole moment equation)
 2. Generate synthetic data using TomoFast-x forward calculation
 3. Run TomoFast-x 3D inversion (WSL or Linux)
 4. Extract best-fit dipole 3D location for each grain (anomaly)
 5. Compute dipole moment, declination, and inclination


 Every step in an independent tool
```

# 1- Create Synthetic Data

#### VIP_Note Conversion rule between Tomofast and QDM

**QDM:** Z-axis positive *up*  
**Tomofast:** Z-axis positive *down*

| Conversion | Rule |
|-----------|------|
| **QDM → Tomofast** | Flip both **Bz** and **Mz** |
| **Tomofast → QDM** | Flip both **Bz** and **Mz** |
| **Tomofast → Tomofast** | Don’t flip **Bz** and don’t flip **Mz** |
| **QDM → QDM** | Don’t flip **Bz** and don’t flip **Mz** |


In [ ]:
# Create Folder

import os

base_dir = os.getcwd()
folder_path = os.path.join(base_dir, "1-Synthetic_Data")

if os.path.exists(folder_path):
    print("Folder already exists:", folder_path)
else:
    os.mkdir(folder_path)
    print("Folder created:", folder_path)



In [ ]:
# The line creates synthetic data using user defined inputs (X, Y, Z, I, D, Amp)

import numpy as np
import matplotlib.pyplot as plt
import scipy.io as sio
import pandas as pd
import os


# Force full array printing
#np.set_printoptions(threshold=np.inf)

# Get the absolute folder path where this script is located
base_dir = os.getcwd()  # current working directory


# -----------------------------
# Constants & grid
# -----------------------------
MU0_OVER_4PI = 1e-7  # T·m/A

# define x , y , and spacing
x_um = np.arange(0, 150+1, 1)
y_um = np.arange(0, 150+1, 1)
X, Y = np.meshgrid(x_um, y_um)
sensor_z_um = -5

# Flatten grid (for field calculation)
Xf = X.ravel() * 1e-6
Yf = Y.ravel() * 1e-6
Zf = np.full_like(Xf, sensor_z_um * 1e-6)
Bz_total = np.zeros_like(Xf, dtype=float)

# storage for any number of dipoles ----
mx_list, my_list, mz_list = [], [], []
Mx_list, My_list, Mz_list = [], [], []   # Mz will be stored flipped (Tomofast convention)

# =============================
#  Dipole 1  
# =============================
xd_um, yd_um, zd_um = 50, 75, -0.5
inc_deg, dec_deg, amplitude = 0, 0, 2.55e-16
inc = np.radians(inc_deg); dec = np.radians(dec_deg)
h = amplitude * np.cos(inc)
mx = h * np.sin(dec); my = h * np.cos(dec); mz = -amplitude * np.sin(inc)
xd, yd, zd = xd_um*1e-6, yd_um*1e-6, zd_um*1e-6
rx, ry, rz = Xf - xd, Yf - yd, Zf - zd
r = np.sqrt(rx*rx + ry*ry + rz*rz); r = np.where(r == 0, np.finfo(float).eps, r)
rhat_x, rhat_y, rhat_z = rx/r, ry/r, rz/r
mdotr = mx*rhat_x + my*rhat_y + mz*rhat_z
Bz_total += MU0_OVER_4PI * (3.0*mdotr*rhat_z - mz) / (r**3)

print("\n--- Dipole 1 ---")
print("mx=", mx, "my=", my, "mz=", mz)
#print("Bz3 (T)", Bz_total)

# ---- : store dipole  moments ----
mx_list.append(mx)
my_list.append(my)
mz_list.append(mz)

#convert from moment to magnetization vector by dividing by voxel (cell) volume 
dx_micro = 1e-6
dy_micro = 1e-6
dz_micro = 1e-6
V_micro = dx_micro * dy_micro * dz_micro
Mx = mx / V_micro
My = my / V_micro
Mz = mz / V_micro

print("\n--- Note Mz are flipped from qdm to tomofast convention positive down ---")
print("\n--- Dipole 1---")
print("Mx(meters)=", Mx, "My(meters)=", My, "Mz(meters)=", -Mz)

# store dipole magnetization (with flipped Mz, as printed) ----
Mx_list.append(Mx)
My_list.append(My)
Mz_list.append(-Mz)


# =============================
#  Dipole 2  
# =============================
xd_um, yd_um, zd_um = 50, 75, -30.5
inc_deg, dec_deg, amplitude = 0, 0, 2.55e-16
inc = np.radians(inc_deg); dec = np.radians(dec_deg)
h = amplitude * np.cos(inc)
mx = h * np.sin(dec); my = h * np.cos(dec); mz = -amplitude * np.sin(inc)
xd, yd, zd = xd_um*1e-6, yd_um*1e-6, zd_um*1e-6
rx, ry, rz = Xf - xd, Yf - yd, Zf - zd
r = np.sqrt(rx*rx + ry*ry + rz*rz); r = np.where(r == 0, np.finfo(float).eps, r)
rhat_x, rhat_y, rhat_z = rx/r, ry/r, rz/r
mdotr = mx*rhat_x + my*rhat_y + mz*rhat_z
Bz_total += MU0_OVER_4PI * (3.0*mdotr*rhat_z - mz) / (r**3)

print("\n--- Dipole 2 ---")
print("mx=", mx, "my=", my, "mz=", mz)
#print("Bz3 (T)", Bz_total)

# ---- : store dipole  moments ----
mx_list.append(mx)
my_list.append(my)
mz_list.append(mz)

#convert from moment to magnetization vector by dividing by voxel (cell) volume 
dx_micro = 1e-6
dy_micro = 1e-6
dz_micro = 1e-6
V_micro = dx_micro * dy_micro * dz_micro
Mx = mx / V_micro
My = my / V_micro
Mz = mz / V_micro

print("\n--- Note Mz are flipped from qdm to tomofast convention positive down ---")
print("\n--- Dipole 2---")
print("Mx(meters)=", Mx, "My(meters)=", My, "Mz(meters)=", -Mz)

# store dipole magnetization (with flipped Mz, as printed) ----
Mx_list.append(Mx)
My_list.append(My)
Mz_list.append(-Mz)



# -----------------------------
# Finish: reshape and noise
# -----------------------------
Bz_T = Bz_total.reshape(X.shape)
Bz_nT = Bz_T * 1e9 # To nanoTesla
rng = np.random.default_rng(42)
Bz_nT = Bz_nT #+ 400.0 + rng.normal(0, 50, size=Bz_nT.shape) # add noise (not recommended)

# -----------------------------
# Plot
# -----------------------------
plt.figure(figsize=(7,6))
vmax = np.nanmax(np.abs(Bz_nT))
im = plt.pcolormesh(X, Y, Bz_nT, shading='auto', cmap='seismic',vmin=-vmax, vmax=vmax) # vmin=-vmax, vmax=vmax)vmin=-1, vmax=1)
plt.colorbar(im, label='Bz (nT)')
plt.xlabel('x (µm)')
plt.ylabel('y (µm)')
plt.tight_layout()
plt.savefig(os.path.join(base_dir,"1-Synthetic_Data", "dipole_field_Bz.png"), dpi=300)   # <-- save plot as PNG
plt.show()

# -----------------------------
# Save data to .CSV and .MAT
# -----------------------------
# ===================
# Flatten for output
x_out = X.ravel()
y_out = Y.ravel()
z_out = np.full_like(x_out, sensor_z_um)
bz_out = Bz_nT.ravel()

# --- CSV ---
df = pd.DataFrame({
    "x_um": x_out,
    "y_um": y_out,
    "z_um": z_out,
    "Bz_nT": bz_out
})
df.to_csv(os.path.join(base_dir, "1-Synthetic_Data","dipole_field_Bz.csv"), index=False)

# ----- Make a MATLAB-style "Bz.mat" output  -----
# Note: Units in the QDMLAB MATLAB  are Tesla and we use nT for our QDM 
Bz = (Bz_nT * 1e-9).astype(float)  # convert nT -> T
Bu = Bz.copy()                      # test stub: use Bz as Bu
Bt = np.abs(Bz)                     # test stub: |Bz| as total field

# Geometry/metadata
step = float(np.diff(x_um).mean() * 1e-6)   # pixel size in meters
h = float(sensor_z_um * 1e-6)               # NV-sample distance in meters
Ny, Nx = Bz.shape
corners = np.array([[1, 1], [Ny, Nx]], dtype=np.int32)  # full image (MATLAB 1-based indexing)
newLED = np.linspace(0, 1, Bz_nT.size).reshape(Bz_nT.shape) * 1e-3

# Save with the same variable names the MATLAB code writes
matlab_like_path = os.path.join(base_dir, "1-Synthetic_Data", "dipole_field_Bz_4dipoles_Sep_35.2Mm.mat")
sio.savemat(matlab_like_path, {
    "Bz": Bz,
    "Bt": Bt,
    "Bu": Bu,
    "h": h,
    "step": step,
    "corners": corners,
    "newLED": newLED,
})
print("Wrote:", matlab_like_path)

#  write TXT files for moments and magnetization ----
n_dip = len(mx_list)

moments_path = os.path.join(base_dir, "1-Synthetic_Data", "dipoles_moments.txt")
mag_path     = os.path.join(base_dir,"1-Synthetic_Data", "dipoles_magnetization.txt")

# 1) mx my mz (original moments)
with open(moments_path, "w") as f:
    f.write(f"{n_dip}\n")  # header: number of cells (dipoles)
    for a, b, c in zip(mx_list, my_list, mz_list):
        f.write(f"{a:.16e} {b:.16e} {c:.16e}\n")

# 2) Mx My Mz (magnetization, with Mz already flipped above)
with open(mag_path, "w") as f:
    f.write(f"{n_dip}\n")  # header: number of cells (dipoles)
    for a, b, c in zip(Mx_list, My_list, Mz_list):
        f.write(f"{a:.16e} {b:.16e} {c:.16e}\n")

print("Wrote:", moments_path)
print("Wrote:", mag_path)


In [ ]:
import numpy as np
import os

base_dir = os.getcwd()

# -----------------------------
# User inputs
# -----------------------------
num_dipoles = 2   # how many dipoles (rows after header) to merge/sum
d = 1             # cell size in microns (dx=dy=dz=d), matches original voxel definition

mag_path = os.path.join(base_dir, "1-Synthetic_Data", "dipoles_magnetization.txt")
out_path = os.path.join(base_dir, "1-Synthetic_Data", "merged_dipoles_summary.txt")

# -----------------------------
# Read magnetization file
# -----------------------------
with open(mag_path, "r") as f:
    lines = f.readlines()

Nmodel = int(lines[0].strip())
data = np.loadtxt(lines[1:])
data = np.atleast_2d(data)

n = min(num_dipoles, Nmodel)
Mx_sum, My_sum, Mz_sum = data[:n].sum(axis=0)

# -----------------------------
# Vector direction calculation (same formula as before)
# -----------------------------
dx = dy = dz = d * 1e-6
V = dx * dy * dz

m = np.sqrt(Mx_sum**2 + My_sum**2 + Mz_sum**2)

if m == 0:
    dec, inc = np.nan, np.nan
else:
    theta = np.degrees(np.arccos(Mz_sum / m))
    phi = np.degrees(np.arctan2(My_sum, Mx_sum))

    inc = theta - 90
    dec = 90 - phi

    change = True
    while change:
        change = False
        if inc > 90:
            inc = 180 - inc
            dec = dec + 180
            change = True
        if inc < -90:
            inc = -180 - inc
            dec = dec + 180
            change = True
        if dec < 0:
            dec = dec + 360
            change = True
        if dec >= 360:
            dec = dec - 360
            change = True

moment = m * V

# -----------------------------
# Print + write result
# -----------------------------
print(f"Merged {n} dipoles -> dec={dec:.4f}, inc={inc:.4f}, moment={moment:.6e} A.m^2")

with open(out_path, "w") as f:
    f.write("num_dipoles_merged dec inc moment(A.m2)\n")
    f.write(f"{n} {dec:.6f} {inc:.6f} {moment:.16e}\n")

print("Wrote:", out_path)

In [ ]:
# Vector Direction Calculation (Double Check Step)
import os
import numpy as np

# --- paths ---
base_dir  = os.getcwd()  # or whatever you already use
mag_path  = os.path.join(base_dir, "1-Synthetic_Data", "dipoles_magnetization.txt")

# --- read Mx, My, Mz from file (skip first line = number of vectors) ---
data = np.loadtxt(mag_path, skiprows=1)   # shape: (N, 3) or (3,) if only one row
if data.ndim == 1:
    data = data[None, :]                  # force 2D if single row

# === Declination & Inclination for each row ===
for idx, (Mx, My, Mz) in enumerate(data, start=1):

    # ----  original code ----
    Mz = -Mz
    m = np.sqrt(Mx**2 + My**2 + Mz**2)

    if m == 0:
        dec, inc = np.nan, np.nan
    else:
        theta = np.degrees(np.arccos(Mz / m))
        phi   = np.degrees(np.arctan2(My, Mx))

        inc = theta - 90
        #inc = 90 -  theta
        #dec= phi - 90
        dec = 90- phi

        change = True
        while change:
            change = False

            if inc > 90:
                inc = 180 - inc # i' = 180-i
                dec = dec + 180 # d' = d+180
                change = True

            if inc < -90:
                inc = -180 - inc # i' = -180-i
                dec = dec + 180 #d' = d+180
                change = True

            if dec < 0:
                dec = dec + 360 #d' = d+360
                change = True

            if dec >= 360:
                dec = dec - 360 #d' = d-360
                change = True

    print(f"Vector {idx}:")
    if m != 0:
        print("  phi=", phi,   "theta=", theta)
    print("  dec=", dec, "inc=", inc)


# 2- Tomofast Synthetic Data (Forward Calculation)

In [ ]:
# Create Folder
import os

base_dir = os.getcwd()
folder_path = os.path.join(base_dir, "2-Forward_Data") 

if os.path.exists(folder_path):
    print("Folder already exists:", folder_path)
else:
    os.mkdir(folder_path)
    print("Folder created:", folder_path)


### 1. Data Prep

In [ ]:
#Data Perp. full scan ( or clip optional)
import os
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Output mode: "bz" for Bz-only (4-column), "bxyz" for full vector (6-column)
# ------------------------------------------------------------
OUTPUT_MODE = "bxyz"   # change to "bxyz" when running the vector workflow


base_dir = os.getcwd()  # current working directory
    
def plot_data(x_coords, y_coords, values, title, out_dir= os.path.join(base_dir, "2-Forward_Data")):
    # Make sure output directory exists
    os.makedirs(out_dir, exist_ok=True)

    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(x_coords, y_coords, c=values, cmap='jet', s=1)
    #plt.axis('equal')
    plt.colorbar(scatter, label='Bz [nT]')
    plt.xlabel('x [µm]')
    plt.ylabel('y [µm]')
    #plt.title(title)

    # Save plot to file
    filename = os.path.join(out_dir, f"{title.replace(' ', '_')}.png")
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    print(f"Plot saved to {filename}")

    plt.show()
    
#filename_z = os.path.join(base_dir, "I=90", "data", "mag_calc_prior_data.txt")
#data_z = np.loadtxt(filename_z, skiprows=1)

filename_z = os.path.join(base_dir, "1-Synthetic_Data", "dipole_field_Bz.csv")
data_z = np.loadtxt(filename_z, delimiter=",", skiprows=1)

print(data_z.shape)

x_coords = data_z[:, 0]
y_coords = data_z[:, 1]
z_coords = data_z[:, 2]
#z_coords = 0*data_z[:, 2] #Make Z=0

bz = data_z[:, 3]

bz = -1*data_z[:, 3] #Flipping Bz for tomofast coordiante system Z is positive down, the QDM Bz is positive up

#bx = data_z[:, 3]
#by = data_z[:, 4]

#amplitude
#b_mag = np.sqrt(bx**2 + by**2 + bz**2)


# Plot the image of data.
plot_data(x_coords, y_coords, bz, 'bz')

#==============================================================
# Find unique coordinates to determine grid dimensions.
unique_x = np.unique(x_coords)
unique_y = np.unique(y_coords)

nx = len(unique_x)  # number of points in X direction
ny = len(unique_y)  # number of points in Y direction

print("Grid dimensions:", nx, ny)

x_coords = x_coords.reshape(ny, nx)
y_coords = y_coords.reshape(ny, nx)
z_coords = z_coords.reshape(ny, nx)

bz = bz.reshape(ny, nx)
#==============================================================
# Clipping the data.
#enter the cell index (given the one index equal 4.4 increment)
# to cover the full range (min → max)
x_start = 0
x_end   = nx
y_start = 0
y_end   = ny

x_coords = x_coords[y_start:y_end, x_start:x_end]
y_coords = y_coords[y_start:y_end, x_start:x_end]
z_coords = z_coords[y_start:y_end, x_start:x_end]
bz = bz[y_start:y_end, x_start:x_end]

print(bz.shape)

# Convert back to 1D array.
x_coords = x_coords.flatten()
y_coords = y_coords.flatten()
z_coords = z_coords.flatten()
bz = bz.flatten()

# Plot the image of data.
plot_data(x_coords, y_coords, bz, 'Clipped bz')

print('Xmin, Xmax =', np.min(x_coords), np.max(x_coords))
print('Ymin, Ymax =', np.min(y_coords), np.max(y_coords))

#==============================================================
# Write tomofast data grid.
# Bz-only (4 cols) or full Bxyz (6 cols), depending on OUTPUT_MODE
if OUTPUT_MODE == "bxyz":
    bx = np.zeros_like(bz)
    by = np.zeros_like(bz)
    data_combined = np.column_stack((x_coords, y_coords, z_coords, bx, by, bz))
    fmt = "%f %f %f %f %f %f"
elif OUTPUT_MODE == "bz":
    data_combined = np.column_stack((x_coords, y_coords, z_coords, bz))
    fmt = "%f %f %f %f"
else:
    raise ValueError(f"Unknown OUTPUT_MODE: {OUTPUT_MODE!r} - use 'bz' or 'bxyz'.")

Ndataz = len(bz)
print(f"Ndata = {Ndataz}  (mode = {OUTPUT_MODE})")
np.savetxt(os.path.join(base_dir, "2-Forward_Data", "Mag_Forward.obs"), data_combined, delimiter=' ', fmt=fmt, header=str(Ndataz), comments='')

### 2. Mesh Creation

In [ ]:
#Mesh Creation of Dipoles
import os
import numpy as np
base_dir = os.getcwd()
out_path = os.path.join(base_dir, "2-Forward_Data", "Meshgrid_Forward.txt")
os.makedirs(os.path.dirname(out_path), exist_ok=True)
# =====================================================
# 1) DEFINE  CELLS HERE of each dipole
#    Each tuple: (xmin, xmax, ymin, ymax, zmin, zmax)
#    Cell size = 1 (same centers as the original size-2 cells)
# =====================================================
cells = [
    # dipole 1
    (49.5, 50.5,74.5, 75.5, 0, 1),
   # dipole 2
    (49.5, 50.5, 74.5, 75.5, 30, 31),  
]
# =====================================================
# 2) BUILD LINE DATA (3 data components, i,j,k indices)
# =====================================================
line_data = []
for idx, (xmin, xmax, ymin, ymax, zmin, zmax) in enumerate(cells, start=1):
    # data comps = 0.0, indices = idx, 1, 1 (change if you want)
    line_data.append([
        xmin, xmax,
        ymin, ymax,
        zmin, zmax,
        0.0, 0.0, 0.0,   # 3 data components
        idx, 1, 1        # i, j, k
    ])
line_data = np.array(line_data)
num_cells = line_data.shape[0]
print("num_cells =", num_cells)
# =====================================================
# 3) WRITE TOMOFAST MESH FILE
# =====================================================
with open(out_path, "w") as f:
    f.write(f"{num_cells}\n")
    np.savetxt(
        f,
        line_data,
        fmt="%.6f %.6f %.6f %.6f %.6f %.6f %.6f %.6f %.6f %d %d %d"
    )
print("Saved:", out_path)

### 3. Create Parfile

In [ ]:
# Create Folder
import os

base_dir = os.getcwd()
folder_path = os.path.join(base_dir,  "0-Parfiles")  

if os.path.exists(folder_path):
    print("Folder already exists:", folder_path)
else:
    os.mkdir(folder_path)
    print("Folder created:", folder_path)


In [ ]:
# Create forward parfile
#define num of true dipoles as nx and num of data points as ndata
import os

base_dir = os.getcwd()
output_path = os.path.join(base_dir,"0-Parfiles", "Parfile_QDM_Forward.txt")

par_text = """==================================================================================
GLOBAL
==================================================================================
global.outputFolderPath         = ./output/QDM
global.description              = QDM inversion data

==================================================================================
MODEL GRID parameters
==================================================================================
# nx ny nz
modelGrid.size                        = 2 1 1
modelGrid.magn.file                   = data/QDM/Meshgrid_Forward.txt
modelGrid.magn.nModelComponents       = 3
==================================================================================
DATA parameters
==================================================================================
forward.data.magn.nData                = 22801  
forward.data.magn.dataGridFile         = data/QDM/Mag_Forward.obs
forward.data.magn.dataValuesFile       = data/QDM/Mag_Forward.obs
forward.data.magn.nDataComponents      = 3  # 1 for bz only and 3 for bxyz

==================================================================================
MAGNETIC FIELD constants
==================================================================================
forward.magneticField.inclination              = 90.0
forward.magneticField.declination              = 0.0
forward.magneticField.intensity_nT             = 53829. 
forward.magneticField.XaxisDeclination         = 0.d0

==================================================================================
DEPTH WEIGHTING
==================================================================================
# 1-depth weight, 2-distance weight.
forward.depthWeighting.type                    = 1
forward.depthWeighting.magn.Z0             = 5.0 #adjust z0 paramter to data height level (-5)


==================================================================================
SENSITIVITY KERNEL
==================================================================================
# 0-calculate the sensitivity, 1-read from files at the specified folder.
sensit.readFromFiles                  = 0
sensit.folderPath                     = ./SENSIT/

==================================================================================
MATRIX COMPRESSION
==================================================================================
# 0-none, 1-wavelet compression.
forward.matrixCompression.type                        = 0
# The minimum compressed sensitivity absolute value.
forward.matrixCompression.rate                        = 0.02

==================================================================================
PRIOR MODEL
==================================================================================
# 1-set value, 2-read from file.
inversion.priorModel.type             = 2
inversion.priorModel.magn.file        = data/QDM/dipoles_magnetization.txt

==================================================================================
INVERSION parameters
==================================================================================
inversion.nMajorIterations              = 0
inversion.nMinorIterations              = 100

==================================================================================
MODEL DAMPING (m - m_prior)
==================================================================================
inversion.modelDamping.magn.weight      = 0.0

==================================================================================
JOINT INVERSION parameters
==================================================================================
inversion.joint.grav.problemWeight      = 0.00000000
inversion.joint.magn.problemWeight      = 1.00000000

==================================================================================
DISJOINT INTERVAL BOUND constraints (ADMM)
==================================================================================
inversion.admm.enableADMM               = 0
inversion.admm.nLithologies             = 1
inversion.admm.magn.bounds              = 0. 1.
inversion.admm.magn.weight              = 100.

inversion.admm.dataCostThreshold        = 3.e-3
inversion.admm.weightMultiplier         = 2.0
inversion.admm.maxWeight                = 1.e+6
"""

with open(output_path, "w") as f:
    f.write(par_text)

print("Parfile created:", output_path)


### 3. Tomofast Run

In [ ]:
### Run Tomofast Forward Data

# Before Running this line check the Forward Parfile data number Ndata (In the data prep line above) and the model data number (dipoles cells)
# For ubuntu don't use dir_wsl and work on base dir and remove "bash" from run lines 

import subprocess
import sys
import os

# ===== USER DEFINED TOMOFAST-x DIRECTORY =====
# Change this ONCE for each machine
TOMOFAST_HOME_WSL = "/home/abekhit/Tomofast-X"


# Detect current Jupyter directory (where the files are)
base_dir = os.getcwd()
print("Current working directory (Windows):", base_dir)

# Convert Windows path (e.g. C:\Users\USER\...) to WSL path (/mnt/c/Users/USER/...)
def to_wsl_path(win_path: str) -> str:
    win_path = win_path.replace("\\", "/")   # C:/Users/USER/OneDrive - Curtin/...
    drive = win_path[0].lower()              # 'c'
    rest = win_path[2:]                      # /Users/USER/OneDrive - Curtin/...
    return f"/mnt/{drive}/{rest}"

base_dir_wsl = to_wsl_path(base_dir)
print("WSL version of current dir:", base_dir_wsl)


WSL = r"C:\Windows\System32\wsl.exe"

# Helper function to run a WSL command with live output
def run_live(cmd):
    process = subprocess.Popen(
        [WSL, "bash", "-lc", cmd],  # If running directly in Ubuntu VM, use cmd only
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    for line in process.stdout:
        print(line, end="")
        sys.stdout.flush()
    process.wait()
    print(f"\n[Return code: {process.returncode}]\n")


print("=== COPYING FILES TO TOMOFAST-x ===\n")

# ---------------------------------------------------------
# COPY COMMANDS
# ---------------------------------------------------------
run_live(f"cp '{base_dir_wsl}/0-Parfiles/Parfile_QDM_Forward.txt'        {TOMOFAST_HOME_WSL}/parfiles/")
run_live(f"cp '{base_dir_wsl}/1-Synthetic_Data/dipoles_magnetization.txt' {TOMOFAST_HOME_WSL}/data/QDM/")
run_live(f"cp '{base_dir_wsl}/2-Forward_Data/Mag_Forward.obs'            {TOMOFAST_HOME_WSL}/data/QDM/")
run_live(f"cp '{base_dir_wsl}/2-Forward_Data/Meshgrid_Forward.txt'      {TOMOFAST_HOME_WSL}/data/QDM/")

print("\n=== RUNNING TOMOFAST-X ===\n")

# Single-core mode in case of single cell or small models or Forward Data
run_live(f"cd {TOMOFAST_HOME_WSL} && ./tomofastx -p ./parfiles/Parfile_QDM_Forward.txt | tee ./output/QDM/QDM_log.txt")

# MPI example (after OpenMPI installed):
# run_live(f"cd {TOMOFAST_HOME_WSL} && mpirun -np 4 ./tomofastx -p ./parfiles/Parfile_QDM_Inversion.txt | tee ./output/QDM/QDM_log.txt")


In [ ]:
# Create Folder
import os

base_dir = os.getcwd()
folder_path = os.path.join(base_dir, "2-Forward_Data", "Tomofast_Output")    

if os.path.exists(folder_path):
    print("Folder already exists:", folder_path)
else:
    os.mkdir(folder_path)
    print("Folder created:", folder_path)


In [ ]:
# Copy Outputs 

# === COPY TOMOFAST OUTPUT FROM LINUX  -> WINDOWS ===

# Windows folder to receive Tomofast output
output_win = os.path.join(base_dir, "2-Forward_Data","Tomofast_Output")  
os.makedirs(output_win, exist_ok=True)

# Convert to WSL path
output_wsl = to_wsl_path(output_win)

print("Copying Tomofast-x output from Linux → Windows folder:")
print("Linux source:   /home/abekhit/Tomofast-x/output/QDM/")
print("Windows target:", output_win)


run_live(f"cp -r  {TOMOFAST_HOME_WSL}/output/QDM/data '{output_wsl}/'")
run_live(f"cp -r  {TOMOFAST_HOME_WSL}/output/QDM/model '{output_wsl}/'")
run_live(f"cp -r  {TOMOFAST_HOME_WSL}/output/QDM/Paraview '{output_wsl}/'")
run_live(f"cp -r  {TOMOFAST_HOME_WSL}/output/QDM/QDM_log.txt '{output_wsl}/'")
run_live(f"cp -r  {TOMOFAST_HOME_WSL}/output/QDM/Parfile_copy.txt '{output_wsl}/'")
run_live(f"cp -r  {TOMOFAST_HOME_WSL}/output/QDM/costs.txt '{output_wsl}/'")
#run_live(f"cp -r  {TOMOFAST_HOME_WSL}/output/QDM/SENSIT '{output_wsl}/'") #Very large and not needed




### 4. Data Prep (Output)

In [ ]:
#Data Perp. full scan ( or clip optional)
# Note this will overwrite the Synthetic Data which used to create the data ground
# Automatically handles either 4-column (x y z Bz) or 6-column (x y z Bx By Bz) mag_prior.txt
import os
import numpy as np
import matplotlib.pyplot as plt
base_dir = os.getcwd()  # current working directory


def plot_data(x_coords, y_coords, values, title, label='Bz [nT]', out_dir=os.path.join(base_dir, "2-Forward_Data")):
    os.makedirs(out_dir, exist_ok=True)
    plt.figure(figsize=(10, 8))
    vmax = np.nanmax(np.abs(values))
    vmin = -vmax
    scatter = plt.scatter(x_coords, y_coords, c=values, cmap='seismic', s=3, vmin=vmin, vmax=vmax)
    plt.axis('equal')
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    cbar = plt.colorbar(scatter)
    cbar.set_label(label, fontsize=14)
    cbar.ax.tick_params(labelsize=12)
    plt.xlabel('x [µm]', fontsize=14)
    plt.ylabel('y [µm]', fontsize=14)
    plt.margins(0)

    filename = os.path.join(out_dir, f"{title.replace(' ', '_')}.png")
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    print(f"Plot saved to {filename}")
    plt.show()


filename_z = os.path.join(base_dir, "2-Forward_Data", "Tomofast_Output", "data", "mag_prior.txt")
data_z = np.loadtxt(filename_z, skiprows=1)
print(data_z.shape)

ncols = data_z.shape[1]
if ncols == 6:
    OUTPUT_MODE = "bxyz"
elif ncols == 4:
    OUTPUT_MODE = "bz"
else:
    raise ValueError(f"Unexpected column count in mag_prior.txt: {ncols} (expected 4 or 6).")

print(f"Detected {ncols} columns -> mode = {OUTPUT_MODE}")

x_coords = data_z[:, 0]
y_coords = data_z[:, 1]
z_coords = 0 * data_z[:, 2]  # Make Z = 0

if OUTPUT_MODE == "bxyz":
    bx = data_z[:, 3]
    by = data_z[:, 4]
    bz = data_z[:, 5]
else:
    bz = data_z[:, 3]

# Plot Bz (always available in either mode)
plot_data(x_coords, y_coords, bz, 'bz', label='Bz [nT]')
if OUTPUT_MODE == "bxyz":
    plot_data(x_coords, y_coords, bx, 'bx', label='Bx [nT]')
    plot_data(x_coords, y_coords, by, 'by', label='By [nT]')

#==============================================================
# Find unique coordinates to determine grid dimensions.
unique_x = np.unique(x_coords)
unique_y = np.unique(y_coords)
nx = len(unique_x)
ny = len(unique_y)
print("Grid dimensions:", nx, ny)

x_coords = x_coords.reshape(ny, nx)
y_coords = y_coords.reshape(ny, nx)
z_coords = z_coords.reshape(ny, nx)
bz = bz.reshape(ny, nx)
if OUTPUT_MODE == "bxyz":
    bx = bx.reshape(ny, nx)
    by = by.reshape(ny, nx)

#==============================================================
# Clipping the data.
# to cover the full range (min -> max)
x_start = 0
x_end   = nx
y_start = 0
y_end   = ny

x_coords = x_coords[y_start:y_end, x_start:x_end]
y_coords = y_coords[y_start:y_end, x_start:x_end]
z_coords = z_coords[y_start:y_end, x_start:x_end]
bz = bz[y_start:y_end, x_start:x_end]
if OUTPUT_MODE == "bxyz":
    bx = bx[y_start:y_end, x_start:x_end]
    by = by[y_start:y_end, x_start:x_end]
print(bz.shape)

# Convert back to 1D array.
x_coords = x_coords.flatten()
y_coords = y_coords.flatten()
z_coords = z_coords.flatten()
bz = bz.flatten()
if OUTPUT_MODE == "bxyz":
    bx = bx.flatten()
    by = by.flatten()

plot_data(x_coords, y_coords, bz, 'Clipped bz', label='Bz [nT]')
if OUTPUT_MODE == "bxyz":
    plot_data(x_coords, y_coords, bx, 'Clipped bx', label='Bx [nT]')
    plot_data(x_coords, y_coords, by, 'Clipped by', label='By [nT]')

print('Xmin, Xmax =', np.min(x_coords), np.max(x_coords))
print('Ymin, Ymax =', np.min(y_coords), np.max(y_coords))

#==============================================================
# Write tomofast data grid: 4-col (Bz-only) or 6-col (Bxyz), matching input.
if OUTPUT_MODE == "bxyz":
    data_combined = np.column_stack((x_coords, y_coords, z_coords, bx, by, bz))
    fmt = "%f %f %f %f %f %f"
else:
    data_combined = np.column_stack((x_coords, y_coords, z_coords, bz))
    fmt = "%f %f %f %f"

Ndataz = len(bz)
print(f"Ndata = {Ndataz}  (mode = {OUTPUT_MODE})")
np.savetxt(os.path.join(base_dir, "2-Forward_Data", "Mag_Forward.obs"), data_combined, delimiter=' ', fmt=fmt, header=str(Ndataz), comments='')

# 3- Add noise to the forward data

In [ ]:
# ------------------------------------------------------------
# Add QDM noise + background field to Tomofast mag_prior data,
# looping over each body (same body-loop style as the voxel/glyph plots).
# Uses qdm_noise() from qdm_noise_utils.py (Souza-Junior & Uieda, 2023).
# Original mag_prior.txt is NEVER overwritten - each noise level is saved
# to its own file, e.g. mag_prior_noise_5.txt, mag_prior_noise_50.txt
# ------------------------------------------------------------
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from qdm_noise_utils import qdm_noise

# ------------------------------------------------------------
#   USER PARAMETERS
# ------------------------------------------------------------
STD = 5        # noise standard deviation (nT)
BACKGROUND = 0 #400  # constant background field added on top (nT)

base_dir = os.getcwd()
TOMOFAST_OUTPUT = os.path.join(base_dir, "2-Forward_Data")

print("Current working directory:", base_dir)
print("Looking for Tomofast_Output at:", TOMOFAST_OUTPUT)
print("  -> exists:", os.path.isdir(TOMOFAST_OUTPUT))
if not os.path.isdir(TOMOFAST_OUTPUT):
    raise FileNotFoundError(
        f"TOMOFAST_OUTPUT does not exist: {TOMOFAST_OUTPUT}\n"
        "Check that your notebook's working directory (os.getcwd()) is the folder "
        "that CONTAINS 'Tec_files_bellon', or edit TOMOFAST_OUTPUT above to an absolute path."
    )


def plot_difference(x_coords, y_coords, data_clean, data_noisy, sfx, out_dir, ncols):
    """
    Plots (noisy - clean) on its OWN color scale. The main Bxyz plots use a
    color scale set by the full field amplitude, which is usually far larger
    than the injected noise+background - so the noise is invisible there even
    though it's genuinely present in the data. This isolates just the change.
    """
    os.makedirs(out_dir, exist_ok=True)
    field_cols = [3, 4, 5] if ncols >= 6 else [3]
    labels = ["Bx", "By", "Bz"] if ncols >= 6 else ["Bz"]

    fig, axes = plt.subplots(1, len(field_cols), figsize=(6 * len(field_cols), 6))
    if len(field_cols) == 1:
        axes = [axes]
    fig.suptitle(f"{sfx}: noise + background actually injected (noisy - clean)", fontsize=13)

    x_r = np.round(x_coords, 9)
    y_r = np.round(y_coords, 9)

    for ax, col, component in zip(axes, field_cols, labels):
        diff = data_noisy[:, col] - data_clean[:, col]
        df = pd.DataFrame({"x": x_r, "y": y_r, "b": diff})
        df = df.sort_values(["y", "x"]).reset_index(drop=True)
        unique_x = np.sort(df["x"].unique())
        unique_y = np.sort(df["y"].unique())
        nx, ny = len(unique_x), len(unique_y)
        if len(df) != nx * ny:
            print(f"[SKIP diff] {sfx} {component}: not a regular grid")
            continue
        X = df["x"].values.reshape(ny, nx)
        Y = df["y"].values.reshape(ny, nx)
        Z = df["b"].values.reshape(ny, nx)
        scale = np.nanmax(np.abs(Z))  # scaled to the DIFFERENCE itself, not the full field
        im = ax.contourf(X, Y, Z, levels=100, cmap="seismic", vmin=-scale, vmax=scale)
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label(f"{component} diff (nT)", fontsize=12)
        ax.set_xlabel("x ($\\mu$m)", fontsize=12)
        ax.set_ylabel("y ($\\mu$m)", fontsize=12)
        ax.set_title(f"{component} (mean={Z.mean():.1f} nT)", fontsize=13)
        ax.set_aspect("equal")
        ax.ticklabel_format(useOffset=False, style="plain")
        ax.xaxis.set_major_locator(plt.MaxNLocator(4))
        ax.yaxis.set_major_locator(plt.MaxNLocator(4))

    plt.tight_layout()
    filename = os.path.join(out_dir, f"Bxyz_{sfx}_diff.png")
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()
    print("Saved:", filename)


def plot_bxyz(x_coords, y_coords, data, sfx, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    ncols = data.shape[1]
    field_cols = [3, 4, 5] if ncols >= 6 else [3]
    labels = ["Bx", "By", "Bz"] if ncols >= 6 else ["Bz"]

    fig, axes = plt.subplots(1, len(field_cols), figsize=(6 * len(field_cols), 6))
    if len(field_cols) == 1:
        axes = [axes]
    fig.suptitle(sfx, fontsize=14)

    bz_col = data[:, field_cols[-1]]
    scale = np.nanmax(np.abs(bz_col))

    for ax, col, component in zip(axes, field_cols, labels):
        b_values = -data[:, col] if component == "Bz" else data[:, col]  # flip Bz, tomofast->qdm convention
        df = pd.DataFrame({"x": np.round(x_coords, 9), "y": np.round(y_coords, 9), "b": b_values})
        df = df.sort_values(["y", "x"]).reset_index(drop=True)
        unique_x = np.sort(df["x"].unique())
        unique_y = np.sort(df["y"].unique())
        nx, ny = len(unique_x), len(unique_y)
        if len(df) != nx * ny:
            print(f"[SKIP] {sfx} {component}: not a regular grid")
            continue
        X = df["x"].values.reshape(ny, nx)
        Y = df["y"].values.reshape(ny, nx)
        Z = df["b"].values.reshape(ny, nx)
        im = ax.contourf(X, Y, Z, levels=100, cmap="seismic", vmin=-scale, vmax=scale)
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label(f"{component} (nT)", fontsize=12)
        ax.set_xlabel("x (µm)", fontsize=12)
        ax.set_ylabel("y (µm)", fontsize=12)
        ax.set_title(component, fontsize=13)
        ax.set_aspect("equal")
        ax.ticklabel_format(useOffset=False, style="plain")
        ax.xaxis.set_major_locator(plt.MaxNLocator(4))
        ax.yaxis.set_major_locator(plt.MaxNLocator(4))

    plt.tight_layout()
    filename = os.path.join(out_dir, f"Bxyz_{sfx}.png")
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()
    print("Saved:", filename)


all_entries = sorted(os.listdir(TOMOFAST_OUTPUT))
print(f"\nFound {len(all_entries)} entries in Tomofast_Output:", all_entries)

n_processed = 0

for body in all_entries:
    body_folder = os.path.join(TOMOFAST_OUTPUT, body)
    if not os.path.isdir(body_folder):
        print(f"  [skip] {body}: not a directory")
        continue

    # the original, untouched clean file - always the source, never overwritten
    clean_path = os.path.join(body_folder, "data", "mag_prior.txt")

    # noisy output gets its own file per STD value, e.g. mag_prior_noise_5.txt
    out_path = os.path.join(body_folder, "data", f"mag_prior_noise_{STD}.txt")

    if not os.path.isfile(clean_path):
        print(f"  [missing] {body}: no file at {clean_path}")
        continue

    print(f"\n=== {body} ===")
    n_processed += 1

    with open(clean_path) as f:
        header_line = f.readline()
    data_clean = np.atleast_2d(np.loadtxt(clean_path, skiprows=1))
    ncols = data_clean.shape[1]

    x_coords, y_coords = data_clean[:, 0], data_clean[:, 1]
    x_rounded = np.round(x_coords, 9)
    y_rounded = np.round(y_coords, 9)
    nx = len(np.unique(x_rounded))
    ny = len(np.unique(y_rounded))
    print(f"  Detected grid: nx={nx}, ny={ny}, nx*ny={nx*ny}, n_rows={len(data_clean)}")

    data_noisy = data_clean.copy()
    field_idx = [3, 4, 5] if ncols >= 6 else [3]  # Bx,By,Bz if vector format, else Bz only

    for idx in field_idx:
        df = pd.DataFrame({"x": x_rounded, "y": y_rounded, "v": data_clean[:, idx]})
        df_sorted = df.sort_values(["y", "x"])
        if len(df) != nx * ny:
            print(f"  [WARNING] column {idx}: not a regular grid - noise not added.")
            continue
        grid = df_sorted["v"].to_numpy().reshape(ny, nx)
        noise = qdm_noise(shape=(ny, nx), std_dev=STD)
        noisy_grid = grid + noise + BACKGROUND
        df_sorted = df_sorted.copy()
        df_sorted["v_noisy"] = noisy_grid.flatten()
        data_noisy[:, idx] = df_sorted.sort_index()["v_noisy"].to_numpy()

    # ---- write the noisy version to its OWN file, never touching mag_prior.txt ----
    with open(out_path, "w") as f:
        f.write(header_line if header_line.strip() else f"{data_noisy.shape[0]}\n")
        for row in data_noisy:
            f.write(" ".join(f"{v:.16e}" for v in row) + "\n")
    print(f"  Added noise (std={STD} nT) + background ({BACKGROUND} nT)")
    print("  Original (untouched):", clean_path)
    print("  Wrote noisy version to:", out_path)

    # ---- explicit proof the values actually changed ----
    for idx in field_idx:
        diff = data_noisy[:, idx] - data_clean[:, idx]
        name = {3: "Bx", 4: "By", 5: "Bz"}.get(idx, f"col{idx}")
        print(f"    {name}: mean diff = {diff.mean():.3f} nT (should be close to BACKGROUND={BACKGROUND}), "
              f"identical-to-clean = {np.allclose(diff, 0)}")

    # ---- plot clean vs noisy, to see the difference ----
    sfx = f"{body}_noise_{STD}"
    plot_bxyz(x_coords, y_coords, data_clean, f"{body}_no_noise", body_folder)
    plot_bxyz(x_coords, y_coords, data_noisy, sfx, body_folder)
    plot_difference(x_coords, y_coords, data_clean, data_noisy, sfx, body_folder, ncols)

print(f"\nDone. Processed {n_processed} body folder(s).")
if n_processed == 0:
    print("!!! NOTHING WAS PROCESSED. Check the path/folder listing printed above. !!!")